# Example: Stylized Facts of Log Growth Rate Data

Financial returns exhibit universal statistical patterns — called _stylized facts_ — that deviate from the assumptions of classical models. Understanding these facts is essential for building realistic portfolio models and stress tests. In this example, we compute log growth rates from synthetic training data and examine three key properties: fat-tailed distributions, near-zero autocorrelation of returns, and persistent volatility clustering.

> __Learning Objectives:__
>
> * **Log Growth Rate Computation**: Compute the continuously compounded growth rate (CCGR) matrix from synthetic price data and explore its basic statistical properties
> * **Fat Tails and Non-Normality**: Test whether growth rates follow a Normal or Laplace distribution using the Anderson-Darling test, and estimate the tail index via Hill's estimator
> * **Autocorrelation and Volatility Clustering**: Verify that raw returns are approximately uncorrelated (random walk) while absolute returns exhibit persistent positive autocorrelation (volatility clustering)

## Setup, Data, and Prerequisites
We begin by loading the `eCornellAIFinance` package and the frozen synthetic training dataset via `Include.jl`.

In [ ]:
include("Include.jl");

### Data
We load the frozen 20-year synthetic training dataset generated from the pre-trained JumpHMM portfolio surrogate. This dataset contains daily close prices for 424 tickers plus a synthetic market index, with a curated market path exhibiting realistic drawdowns, jump clusters, and an 8.3% CAGR.

In [ ]:
# load the frozen synthetic training dataset -
training_data = MySyntheticTrainingDataSet();
dataset = training_data["dataset"];
list_of_tickers = training_data["tickers"];  # 424 tickers, sorted alphabetically
println("Loaded $(length(list_of_tickers)) tickers × $(training_data["n_days"]) days ($(training_data["n_years"]) years)")

___
## Task 1: Compute the Log Growth Rate Matrix
We compute the continuously compounded growth rate (CCGR) for every ticker in the dataset. The growth rate of asset $i$ between time $j-1$ and $j$ is:

$$\boxed{g^{(i)}_{j,j-1} = \frac{1}{\Delta t}\ln\frac{S^{(i)}_j}{S^{(i)}_{j-1}}}$$

where $\Delta t = 1/252$ for daily data (one trading day in years).

> __What are we going to do?__ We'll compute the growth rate matrix $\mathbf{X} \in \mathbb{R}^{(T-1) \times N}$ where each row is a time step and each column is a ticker. Then we'll visualize the growth rate time series and its distribution for a selected ticker.
>
> __What should you see?__ The growth rate appears as a stationary random process with non-periodic bursts of extreme volatility followed by calm periods. The distribution has a sharp peak near zero and heavy tails compared to a Normal distribution — this is the "fat tails" stylized fact.

In [ ]:
growth_rate_array = let

    # initialize -
    Δt = 1.0 / 252.0;
    N = length(list_of_tickers);
    T_total = training_data["n_days"];
    T = T_total - 1;

    # compute growth rate matrix: (T-1) × N -
    X = zeros(T, N);
    for (j, ticker) ∈ enumerate(list_of_tickers)
        prices = dataset[ticker][:, :close];
        for t ∈ 1:T
            X[t, j] = (1.0 / Δt) * log(prices[t + 1] / prices[t]);
        end
    end

    # verify dimensions -
    @assert size(X) == (T, N) "Growth rate matrix should be $(T) × $(N)"
    println("Growth rate matrix: $(T) days × $(N) tickers")
    println("Mean growth rate (annualized, all tickers): $(round(mean(X) * 100, digits=2))%")

    X  # return
end

### Visualize
Select a ticker to explore. Try different tickers to see how the growth rate behavior varies across assets.

In [ ]:
ticker_to_visualize = "AAPL";  # change this to explore different tickers
@assert ticker_to_visualize ∈ list_of_tickers "Ticker $(ticker_to_visualize) not in dataset"

**Growth rate time series and distribution:** The left panel shows the daily growth rate over 20 years. The right panel shows the empirical density compared to fitted Normal and Laplace distributions.

> __What should you see?__ The time series shows bursts of high volatility (large positive and negative spikes clustered together) interspersed with calmer periods — this is volatility clustering. The distribution is sharply peaked near zero with heavy tails, matching a Laplace distribution much better than a Normal.

In [ ]:
let
    i = findfirst(==(ticker_to_visualize), list_of_tickers);
    X = growth_rate_array[:, i];

    # left: time series -
    p1 = plot(X, label="", lw=1, color=:steelblue, alpha=0.7,
        xlabel="Trading Day", ylabel="Growth Rate (1/yr)",
        title="$(ticker_to_visualize) Daily Growth Rate");

    # right: density comparison -
    d_normal = fit_mle(Normal, X);
    d_laplace = fit_mle(Laplace, X);

    p2 = plot(d_normal, lw=2, color=:deepskyblue, label="Normal (MLE)",
        xlabel="Growth Rate (1/yr)", ylabel="Density",
        title="$(ticker_to_visualize) Distribution");
    plot!(p2, d_laplace, lw=2, color=:gray40, label="Laplace (MLE)");
    density!(p2, X, lw=2, color=:red, label="Observed");

    plot(p1, p2, layout=(1, 2), size=(1000, 400))
end

___
## Task 2: Is the Growth Rate Normally Distributed?
One of the most robust stylized facts of financial returns is that they are **not normally distributed**. Return distributions have fat tails — the density near zero is greater than Normal, and extreme events occur far more often than a Gaussian model predicts.

> __What are we going to do?__ We use the [Anderson-Darling (AD) test](https://en.wikipedia.org/wiki/Anderson%E2%80%93Darling_test) to formally test whether growth rates follow a Normal or Laplace distribution. The AD test is particularly sensitive to deviations in the tails. We also compute Hill's estimator of the tail index $\alpha$, which quantifies how heavy the tails are.
>
> __What should you see?__ The AD test should **reject** the Normal hypothesis (small p-value) but **fail to reject** the Laplace hypothesis (large p-value) for most tickers. Hill's tail index $\alpha$ for equity returns is typically between 2 and 5 — far from the infinite tail index implied by a Normal distribution.

In [ ]:
let
    i = findfirst(==(ticker_to_visualize), list_of_tickers);
    μ = growth_rate_array[:, i];

    # fit distributions -
    d_normal = fit_mle(Normal, μ);
    d_laplace = fit_mle(Laplace, μ);

    # AD tests -
    ad_normal = OneSampleADTest(μ, d_normal);
    ad_laplace = OneSampleADTest(μ, d_laplace);

    println("Anderson-Darling Test Results for $(ticker_to_visualize):")
    println("═"^55)
    println("  Normal:  A² = $(round(ad_normal.A², digits=2)), p-value = $(pvalue(ad_normal) < 1e-6 ? "<1e-6" : string(round(pvalue(ad_normal), digits=4)))")
    println("  Laplace: A² = $(round(ad_laplace.A², digits=2)), p-value = $(round(pvalue(ad_laplace), digits=4))")
    println()
    println("  Normal rejected at α=0.05? $(pvalue(ad_normal) < 0.05 ? "YES ✗" : "NO ✓")")
    println("  Laplace rejected at α=0.05? $(pvalue(ad_laplace) < 0.05 ? "YES ✗" : "NO ✓")")
end

**Hill's Tail Index:** The tail index $\alpha$ quantifies how heavy the tails of the distribution are. For a Normal distribution, $\alpha \to \infty$. For financial returns, $\alpha$ is typically between 2 and 5 — meaning extreme events occur _much_ more often than a Gaussian model predicts.

> __Hill's estimator__ works by sorting the positive observations in descending order and measuring how quickly the tail decays. A smaller $\alpha$ means heavier tails (more extreme events).

In [ ]:
let
    i = findfirst(==(ticker_to_visualize), list_of_tickers);
    μ = growth_rate_array[:, i];

    # Hill's estimator on positive tail -
    pos = filter(>(0), μ);
    n = length(pos);
    z = sort(pos, rev=true);

    # compute: α = 1 / ((1/(n-1)) Σ log(z_i / z_n))
    hill_sum = sum(log(z[i] / z[n]) for i ∈ 1:(n-1));
    α_hill = (n - 1) / hill_sum;

    println("Hill's tail index for $(ticker_to_visualize): α ≈ $(round(α_hill, digits=3))")
    println("  (Normal → ∞, Cauchy → 1, typical equity returns → 2-5)")
end

___
## Task 3: Autocorrelation and Volatility Clustering
Two more stylized facts: (1) raw returns are approximately **uncorrelated** (consistent with the random walk hypothesis), and (2) absolute returns show persistent **positive autocorrelation** — this is volatility clustering, where periods of high volatility tend to follow periods of high volatility.

> __What are we going to do?__ We compute the autocorrelation function (ACF) for both raw growth rates $g_i(t)$ and absolute growth rates $|g_i(t)|$. We compare to the 99% confidence band under the null hypothesis of no autocorrelation.
>
> __What should you see?__ The raw ACF should hover near zero at all lags (random walk). The absolute ACF should be significantly positive for short lags (up to 10–50 days), decaying slowly — this is the volatility clustering signature that motivates models like JumpHMM.

**Autocorrelation of raw returns:** Under the random walk hypothesis, the ACF should be zero for all lags > 0. Significant spikes indicate predictable structure in returns.

> __What should you see?__ Near-zero autocorrelation at all lags, with perhaps a few weak violations in the first few days. This confirms that daily returns are approximately unpredictable — you can't use yesterday's return to forecast today's.

In [ ]:
let
    i = findfirst(==(ticker_to_visualize), list_of_tickers);
    X = growth_rate_array[:, i];
    T = length(X);
    max_lag = 40;
    lags = collect(0:max_lag);

    acf_raw = autocor(X, lags);
    conf_99 = 2.576 / sqrt(T);

    plot(lags, acf_raw, lw=2, color=:steelblue, label="ACF of gₜ ($(ticker_to_visualize))",
        linetype=:sticks, marker=:circle, ms=3,
        xlabel="Lag (trading days)", ylabel="Autocorrelation",
        title="ACF of Raw Returns: $(ticker_to_visualize)", size=(700, 400));
    hline!([conf_99, -conf_99], lw=1.5, ls=:dash, color=:black, label="99% CI")
end

**Volatility clustering (ACF of |gₜ|):** Volatility clustering means that large moves (positive or negative) tend to cluster together. We measure this by computing the ACF of the _absolute_ growth rate $|g_i(t)|$.

> __What should you see?__ Positive autocorrelation that decays slowly over tens or hundreds of lags — far exceeding the 99% confidence bands. This is one of the strongest and most universal stylized facts. It means: if today was volatile, tomorrow is more likely to be volatile too.

In [ ]:
let
    i = findfirst(==(ticker_to_visualize), list_of_tickers);
    X = growth_rate_array[:, i];
    T = length(X);
    max_lag = 100;
    lags = collect(0:max_lag);

    # ACF of absolute returns (not squared — consistent with lecture and surrogate validation) -
    acf_abs = autocor(abs.(X), lags);
    conf_99 = 2.576 / sqrt(T);

    plot(lags, acf_abs, lw=2, color=:red, label="ACF of |gₜ| ($(ticker_to_visualize))",
        linetype=:sticks, marker=:circle, ms=3,
        xlabel="Lag (trading days)", ylabel="Autocorrelation",
        title="Volatility Clustering: ACF of |gₜ| for $(ticker_to_visualize)", size=(700, 400));
    hline!([conf_99, -conf_99], lw=1.5, ls=:dash, color=:black, label="99% CI")
end

___
## Summary

> __Key Takeaways:__
>
> * **Growth rates are not normally distributed**: The Anderson-Darling test rejects the Normal hypothesis for equity growth rates, while the Laplace distribution provides a much better fit — confirming the fat-tails stylized fact that extreme events occur far more often than Gaussian models predict
> * **Raw returns are approximately uncorrelated**: The ACF of $g_i(t)$ drops to near zero at all lags, consistent with the random walk hypothesis — you cannot use past returns to predict future returns
> * **Volatility clusters persistently**: The ACF of $|g_i(t)|$ shows significant positive autocorrelation that decays slowly over tens of days — large moves beget large moves, motivating regime-switching models like JumpHMM that capture this temporal dependence

These stylized facts explain why classical models (which assume i.i.d. Normal returns) fail under stress, and why the JumpHMM surrogate model — which captures fat tails, regime persistence, and volatility clustering — produces more realistic synthetic data for portfolio analysis.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.